For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Benchmarking Semantic Regexes
We compare our semantic regex method to prior natural language feature descriptions. This notebooks contains code to load in the results, plot the metric distrubtions, and run statistical tests.

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import os
import json
import pandas as pd
import altair as alt
from pathlib import Path
import itertools
import numpy as np
import scipy.stats as stats
import brotli

import warnings
warnings.filterwarnings('ignore')
alt.data_transformers.enable("vegafusion")


DataTransformerRegistry.enable('vegafusion')

## Data Loading
Results files are stored on disk. These functions load the metric results for each method and feature.

In [12]:
def import_feature_json(json_file_path, method_map, metric_map):
    """
    Import a feature JSON file (brotli compressed) containing multiple features grouped by layer.

    Parameters:
        json_file_path (str): Path to the brotli compressed JSON file
        method_map (dict): Mapping of method names to display names
        metric_map (dict): Mapping of metric names to column names

    Returns:
        pd.DataFrame: DataFrame with all features from the file
    """
    # Read and decompress brotli file
    with open(json_file_path, 'rb') as f:
        compressed_data = f.read()
        decompressed_data = brotli.decompress(compressed_data)
        group_content = json.loads(decompressed_data)

    rows = []

    # Iterate through each feature in the grouped file
    for feature_key, data in group_content.items():
        # Extract basic feature information
        row = {
            'model_id': data['feature']['model_id'],
            'layer': data['feature']['layer'],
            'index': data['feature']['index'],
            'description': data['description']['description'],
            'method': method_map[data['description']['method']],
        }

        # Extract evaluation metrics using metric_map
        evaluation = data['evaluation']
        for metric_name, metric_plot_name in metric_map.items():
            if metric_name in evaluation and 'value' in evaluation[metric_name]:
                row[metric_plot_name] = evaluation[metric_name]['value']
            else:
                row[metric_plot_name] = None

        rows.append(row)

    # Create DataFrame with all rows from this file
    df = pd.DataFrame(rows)
    return df

In [13]:
def import_all_feature_jsons_multi_folder(base_path, method_map, metric_map, folder_pattern='*'):
    """
    Import all feature brotli files from multiple folders into a single pandas DataFrame.

    Parameters:
        base_path (str): Path to the base directory containing multiple folders
        method_map (dict): Mapping of method names to display names
        metric_map (dict): Mapping of metric names to column names
        folder_pattern (str): Pattern to match folder names (default '*' for all folders)

    Returns:
        pd.DataFrame: Combined DataFrame with all features, including folder and filename columns
    """
    base_path = Path(base_path)
    all_dfs = []

    # Loop through all matching folders
    for folder in base_path.glob(folder_pattern):
        if folder.is_dir():
            # Loop through all brotli files in the current folder
            for brotli_file in folder.glob('*.json.brotli'):
                try:
                    df = import_feature_json(brotli_file, method_map, metric_map)
                    # Add both folder name and filename as columns
                    df['folder_name'] = folder.name
                    df['filename'] = brotli_file.name
                    all_dfs.append(df)
                except Exception as e:
                    raise e

    # Combine all DataFrames
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        return combined_df
    else:
        print("No brotli files found or processed successfully")
        return pd.DataFrame()

In [14]:
def filter_to_common_features(dfs):
    """Filter multiple DataFrames to only include rows with common feature_ids."""
    filtered_dfs = {}
    common_features = None
    for df in dfs.values():
        if common_features is None:
            common_features = set(df['feature_id'])
        else:
            common_features &= set(df['feature_id'])
    for name, df in dfs.items():
        filtered_dfs[name] = df[df['feature_id'].isin(common_features)]
    assert all(len(df) > 0 for df in filtered_dfs.values()), "One of the DataFrames is empty after filtering to common features"
    assert all(len(df) == len(next(iter(filtered_dfs.values()))) for df in filtered_dfs.values()), f"DataFrames have different lengths after filtering to common features: {[len(df) for df in filtered_dfs.values()]}"
    return filtered_dfs

def filter_to_computed_metrics(dfs, metrics):
    filtered_dfs = {}
    scored_features = None
    for _, df in dfs.items():
        if scored_features is None:
            scored_features = set(df.dropna(subset=metrics)['feature_id'])
        else:
            scored_features &= set(df.dropna(subset=metrics)['feature_id'])
    print(f"Number of features with all specified metrics computed: {len(scored_features)}")
    for name, df in dfs.items():
        filtered_dfs[name] = df[df['feature_id'].isin(scored_features)]
    assert all(len(df) > 0 for df in filtered_dfs.values()), "One of the DataFrames is empty after filtering to common features"
    assert all(len(df) == len(next(iter(filtered_dfs.values()))) for df in filtered_dfs.values()), f"DataFrames have different lengths after filtering to common features: {[len(df) for df in filtered_dfs.values()]}"
    return filtered_dfs

def filter_to_n_per_layer(dfs, n=100):
    np.random.seed(42)
    semantic_regex_key = [key for key in dfs.keys() if 'semantic' in key][0]
    semantic_regex_df = dfs[semantic_regex_key]
    num_layers = max(int(layer.split('-')[0]) for layer in semantic_regex_df['layer'].unique()) + 1
    num_per_layer = []
    for layer in range(num_layers):
        layer_df = semantic_regex_df[semantic_regex_df.layer.str.startswith(f'{layer}-')]
        layer_indices = layer_df.index
        random_indices = np.random.permutation(layer_indices)
        semantic_regex_df = semantic_regex_df.drop(random_indices[:-n])
        num_in_layer = semantic_regex_df[semantic_regex_df.layer.str.startswith(f'{layer}-')].shape[0]
        num_per_layer.append(num_in_layer)

    for layer, num_in_layer in enumerate(num_per_layer):
        if num_in_layer < n:
            print(f"Layer {layer} has only {num_in_layer} features after filtering.")

    dfs[semantic_regex_key] = semantic_regex_df
    dfs = filter_to_common_features(dfs)
    return dfs

def load_exp_dfs(experiments, models, method_map, experiments_dir, metric_map):
    experiment_dfs = {}
    for experiment, model, method in itertools.product(experiments, models, method_map.keys()):
        experiment_name = f'{experiment}_{method}_{model}'
        experiment_df = import_all_feature_jsons_multi_folder(experiments_dir, method_map, metric_map, experiment_name)
        print(f"Loaded {len(experiment_df)} features from {experiment_name}")
        # Add a unique feature identifier
        experiment_df['feature_id'] = experiment_df['model_id'].astype(str) + '_' + experiment_df['layer'].astype(str) + '_' + experiment_df['index'].astype(str)
        experiment_dfs[method_map[method]] = experiment_df
    experiment_dfs = filter_to_common_features(experiment_dfs)
    experiment_dfs = filter_to_computed_metrics(experiment_dfs, metric_map.values())
    experiment_dfs = filter_to_n_per_layer(experiment_dfs, n=100)
    print(f"{models[0].upper()}: after loading and filtering, we have {len(experiment_dfs)} experiments with {len(next(iter(experiment_dfs.values())))} common features each.")
    return experiment_dfs

### Load the Experiments

In [15]:
metric_map = {
    'clarity': 'clarity',
    'detection': 'detect',
    'fuzzing': 'fuzzing',
    'responsiveness': 'respon',
    'purity': 'purity',
    'faithfulness': 'faithful',
}
experiments = ['cf1daa']
# methods = [ 'oai_token-act-pair', 'eleuther_acts_top20', 'semantic_regex', ]
method_map = {
    'oai_token-act-pair': 'token-act-pair',
    'eleuther_acts_top20': 'max-acts',
    'semantic_regex': 'semantic-regex',
}
experiments_dir = os.path.join('../', 'artifacts', 'experiments')

In [16]:
all_exp_dfs = {
    'GPT2-RES-25k': load_exp_dfs(experiments, ['gpt2-small_res-jb'], method_map, experiments_dir, metric_map),
    'Gemma-2-2B-RES-16k': load_exp_dfs(experiments, ['gemma-2-2b_gemmascope-res-16k'], method_map, experiments_dir, metric_map),
    'Gemma-2-2B-RES-65k': load_exp_dfs(experiments, ['gemma-2-2b_gemmascope-res-65k'], method_map, experiments_dir, metric_map),
}

Loaded 1837 features from cf1daa_oai_token-act-pair_gpt2-small_res-jb
Loaded 1838 features from cf1daa_eleuther_acts_top20_gpt2-small_res-jb
Loaded 1838 features from cf1daa_semantic_regex_gpt2-small_res-jb
Number of features with all specified metrics computed: 1470
GPT2-SMALL_RES-JB: after loading and filtering, we have 3 experiments with 1300 common features each.
Loaded 4296 features from cf1daa_oai_token-act-pair_gemma-2-2b_gemmascope-res-16k
Loaded 4300 features from cf1daa_eleuther_acts_top20_gemma-2-2b_gemmascope-res-16k
Loaded 25990 features from cf1daa_semantic_regex_gemma-2-2b_gemmascope-res-16k
Number of features with all specified metrics computed: 2949
GEMMA-2-2B_GEMMASCOPE-RES-16K: after loading and filtering, we have 3 experiments with 2600 common features each.
Loaded 4272 features from cf1daa_oai_token-act-pair_gemma-2-2b_gemmascope-res-65k
Loaded 4279 features from cf1daa_eleuther_acts_top20_gemma-2-2b_gemmascope-res-65k
Loaded 25957 features from cf1daa_semantic_reg

In [17]:
gpt4o_metric_map = {
    'clarity': 'clarity',
    'detection': 'detect',
    'fuzzing': 'fuzzing',
    'responsiveness': 'respon',
    'purity': 'purity',}
gpt_4o_exp_dfs = {
    'GPT2-RES-25k_gpt4o': load_exp_dfs(experiments, ['gpt2-small_res-jb_gpt-4o'], method_map, experiments_dir, gpt4o_metric_map),
}

Loaded 2344 features from cf1daa_oai_token-act-pair_gpt2-small_res-jb_gpt-4o
Loaded 2361 features from cf1daa_eleuther_acts_top20_gpt2-small_res-jb_gpt-4o
Loaded 2339 features from cf1daa_semantic_regex_gpt2-small_res-jb_gpt-4o
Number of features with all specified metrics computed: 889
Layer 0 has only 77 features after filtering.
Layer 1 has only 59 features after filtering.
Layer 2 has only 29 features after filtering.
Layer 3 has only 60 features after filtering.
Layer 4 has only 76 features after filtering.
Layer 5 has only 76 features after filtering.
Layer 6 has only 79 features after filtering.
Layer 7 has only 49 features after filtering.
Layer 8 has only 56 features after filtering.
Layer 9 has only 52 features after filtering.
Layer 10 has only 46 features after filtering.
Layer 11 has only 57 features after filtering.
GPT2-SMALL_RES-JB_GPT-4O: after loading and filtering, we have 3 experiments with 816 common features each.


## Compute Quantiative Results
Compute the metric averages and standard deviations.

In [18]:
def compute_metric_avgs_stds(df, metrics):
    """Compute average scores +/- standard deviation for each metric in the DataFrame. Report how many have missing values for each metric."""
    scores = {}
    for metric in metrics:
        assert metric in df.columns, f"Metric '{metric}' not found in DataFrame columns"
        assert not df[metric].empty, f"Metric '{metric}' column is empty"
        avg_score = df[metric].mean() if not df[metric].empty else None
        std_score = df[metric].std() if not df[metric].empty else None
        missing_count = df[metric].isna().sum()
        scores[metric] = {
            'average': avg_score,
            'std_dev': std_score,
            'missing_count': missing_count,
            'total_count': len(df)
        }
    return scores

def print_metric_summary(dfs, metrics):
    for name, df in dfs.items():
        results = compute_metric_avgs_stds(df, metrics)
        print(f"\nExperiment: {name}".upper())
        for metric, result in results.items():
            print(f"{metric}: Average={result['average']}, StdDev={result['std_dev']}, Missing={result['missing_count']}/{result['total_count']}")
        print('& ' + ' & '.join([f'${s["average"]:.2f} \pm {s["std_dev"]:.2f}$' for s in results.values()]) + r' \\') # Print LaTeX table row

In [19]:
# Compute and display average metrics for each experiment
for name, dfs in all_exp_dfs.items():
    print('\n' + '='*80 + '\n')
    print(name.upper())
    print_metric_summary(dfs, metric_map.values())




GPT2-RES-25K

EXPERIMENT: TOKEN-ACT-PAIR
clarity: Average=0.4549307692307693, StdDev=0.3600838817779365, Missing=0/1300
detect: Average=0.7882666666666667, StdDev=0.15130933847477526, Missing=0/1300
fuzzing: Average=0.8055224358974358, StdDev=0.15735617474682542, Missing=0/1300
respon: Average=0.8149515994905396, StdDev=0.230494945764075, Missing=0/1300
purity: Average=0.7162618049445605, StdDev=0.27497146909402725, Missing=0/1300
faithful: Average=0.47164536106121546, StdDev=0.46221325531330637, Missing=0/1300
& $0.45 \pm 0.36$ & $0.79 \pm 0.15$ & $0.81 \pm 0.16$ & $0.81 \pm 0.23$ & $0.72 \pm 0.27$ & $0.47 \pm 0.46$ \\

EXPERIMENT: MAX-ACTS
clarity: Average=0.7094356923076923, StdDev=0.3449254084372251, Missing=0/1300
detect: Average=0.8570698717948718, StdDev=0.11396404698688493, Missing=0/1300
fuzzing: Average=0.8853948717948718, StdDev=0.10158154064462448, Missing=0/1300
respon: Average=0.8696135569485226, StdDev=0.19386667877050948, Missing=0/1300
purity: Average=0.7873118770951

In [20]:
# Compute and display average metrics for each experiment
for name, dfs in gpt_4o_exp_dfs.items():
    print('\n' + '='*80 + '\n')
    print(name.upper())
    print_metric_summary(dfs, gpt4o_metric_map.values())



GPT2-RES-25K_GPT4O

EXPERIMENT: TOKEN-ACT-PAIR
clarity: Average=0.5731867647058824, StdDev=0.365612806720592, Missing=0/816
detect: Average=0.8192473447712417, StdDev=0.1320637128819007, Missing=0/816
fuzzing: Average=0.8139716094771242, StdDev=0.1293910159724911, Missing=0/816
respon: Average=0.873552799451168, StdDev=0.1932738242417522, Missing=0/816
purity: Average=0.8083811677751144, StdDev=0.23384097387003253, Missing=0/816
& $0.57 \pm 0.37$ & $0.82 \pm 0.13$ & $0.81 \pm 0.13$ & $0.87 \pm 0.19$ & $0.81 \pm 0.23$ \\

EXPERIMENT: MAX-ACTS
clarity: Average=0.702445588235294, StdDev=0.33520069121580787, Missing=0/816
detect: Average=0.858719362745098, StdDev=0.09630032876684413, Missing=0/816
fuzzing: Average=0.8508956290849673, StdDev=0.09156614882919437, Missing=0/816
respon: Average=0.8964830106937197, StdDev=0.15428265695292825, Missing=0/816
purity: Average=0.8406381078792762, StdDev=0.20280043385010055, Missing=0/816
& $0.70 \pm 0.34$ & $0.86 \pm 0.10$ & $0.85 \pm 0.09$ & $0.9

## Plot the Metric Distributions

In [21]:
colors = ["#1f77b4", "#17becf", "#d62728"]

In [22]:
def save_chart(chart, filename):
    chart.save(
        filename,
        scale_factor=2,  # Increase scale factor for higher resolution
        format='png',
        ppi=300,
    )

### Box Plots

In [23]:
def plot_metric_box(exp_dfs: dict, metric: str, show_yaxis: bool = False):
    """Plot the each metric as a stacked denstiry plot where each method is a different color using Altair"""
    for exp_name, exp_df in exp_dfs.items():
        exp_df['method'] = exp_name

    combined_df = pd.concat(exp_dfs.values(), ignore_index=True)
    combined_df.dropna(subset=[metric], inplace=True)

    methods = combined_df['method'].unique().tolist()
    sr_method = [m for m in methods if 'regex' in m]
    method_order = [m for m in methods if m not in sr_method] + sr_method

    chart = alt.Chart(combined_df).mark_boxplot(size=10, extent='min-max', opacity=1).encode(
        y=alt.Y(
            f'{metric}:Q',
            scale=alt.Scale(domain=[-0.05, 1.05]),
            title='score' if show_yaxis else None,
            axis=alt.Axis(labels=show_yaxis, ticks=show_yaxis, values=[0.0, 0.5, 1.0], domain=False)
        ),
        x=alt.X(
            'method:N',
            sort=method_order,
            axis=alt.Axis(labels=False, ticks=False, title=f'{metric}', domain=False)
        ),
        color=alt.Color(
            'method:N',
            scale=alt.Scale(domain=method_order, range=colors),
            sort=method_order,
            title="Method"),
    ).properties(
        height=100,
        width=40
    )

    return chart

def plot_metrics_box(exp_dfs, metrics, title=None, hide_yaxis=False):
    """Plot the each metric as a stacked denstiry plot where each method is a different color using Altair"""
    orientation = 'top'
    dy = -5
    if title is None:
        title = 'Metric Distributions per Feature Description Method'

    metric_plots = []
    for i, metric in enumerate(metrics):
        metric_plots.append(plot_metric_box(exp_dfs, metric, show_yaxis=(i==0 and not hide_yaxis)))
    chart = alt.hconcat(*metric_plots, spacing=10).resolve_scale(y="shared").properties(
        title=alt.TitleParams(
            text=title,
            anchor='middle',
            orient=orientation,
            fontSize=16,
            dy=dy  # Move title down a bit
        )
    )

    return chart

def plot_metrics_box_multi(multi_exp_dfs, metrics):
    """Plot the each metric as a stacked denstiry plot where each method is a different color using Altair"""
    plots = []
    for i, (exp_name, exp_dfs) in enumerate(multi_exp_dfs.items()):
        plots.append(plot_metrics_box(exp_dfs, metrics, exp_name, hide_yaxis=i>0))
    plot = alt.hconcat(*plots, spacing=40).properties(
        title='Metric Distributions Per Feature Description Method'
    ).configure_title(
        anchor='middle',
        fontSize=18,
        dy=-5,
    ).configure_axis(
        titleFontSize=12,
        titleColor="grey",
    ).configure_legend(
        orient='none',
        legendX=285,
        legendY=-50,
        direction='horizontal',
        titleBaseline='middle',
        titleOrient='left',
        labelFontSize=14,
        titleFontSize=14
    ).configure_view(
        stroke=None  # Remove the frame around each chart
    )
    return plot


In [24]:
box_plot = plot_metrics_box_multi(all_exp_dfs, metric_map.values())
save_chart(box_plot, 'metric_box_plots.png')
box_plot

alt.HConcatChart(...)

In [25]:
box_plot = plot_metrics_box(gpt_4o_exp_dfs['GPT2-RES-25k_gpt4o'], gpt4o_metric_map.values(), 'Metric Distributions Per Feature Description Method')
save_chart(box_plot, 'metric_box_plots_gpt4o.png')
box_plot

alt.HConcatChart(...)

### Density Plots

In [26]:
def plot_metric_density(exp_dfs: dict, metric: str, show_yaxis: bool = False, overlay: bool = True):
    """Plot the each metric as a stacked denstiry plot where each method is a different color using Altair"""
    for exp_name, exp_df in exp_dfs.items():
        exp_df['method'] = exp_name

    combined_df = pd.concat(exp_dfs.values(), ignore_index=True)
    combined_df.dropna(subset=[metric], inplace=True)

    methods = combined_df['method'].unique().tolist()
    sr_method = [m for m in methods if 'regex' in m]
    method_order = [m for m in methods if m not in sr_method] + sr_method

    y_axis = alt.Y('density:Q', stack=None, axis=alt.Axis(values=[0, 2, 4, 6])) if show_yaxis else alt.Y('density:Q', stack=None, axis=alt.Axis(labels=False, ticks=False, title=None, values=[0, 2, 4, 6]))

    chart = alt.Chart(combined_df).transform_density(
        metric,
        as_=[metric, 'density'],
        groupby=['method'],
        extent=[0, 1]
    ).mark_area(opacity=0.4, interpolate='basis').encode(
        x=alt.X(f'{metric}:Q', scale=alt.Scale(domain=[0, 1])),
        y=y_axis,
        color=alt.Color('method:N', scale=alt.Scale(domain=method_order, range=colors), sort=method_order),
    )

    if not overlay:
        chart = chart.facet(
            facet=alt.Facet('method:N', sort=method_order),
            columns=1)

    chart = chart.properties(
        height=75,
        width=125,
    )

    return chart


def plot_metric_densities(exp_dfs, metrics, overlay=True, title=None):
    """Plot the each metric as a stacked denstiry plot where each method is a different color using Altair"""
    orientation = 'top'
    dy=-5
    if title is None:
        title = 'Metric Distributions Per Feature Description Method'
    metric_density_plots = []
    for i, metric in enumerate(metrics):
        metric_density_plots.append(plot_metric_density(exp_dfs, metric, i==0, overlay))
    chart = alt.hconcat(*metric_density_plots).resolve_scale(y="shared").properties(
        title=alt.Title(
            title,
            anchor='middle',
            orient=orientation,
            dy=dy,
            fontSize=14
        )
    )
    return chart

def plot_metric_densities_multi(multi_exp_dfs, metrics):
    """Plot the each metric as a stacked denstiry plot where each method is a different color using Altair"""
    plots = []
    for exp_name, exp_dfs in multi_exp_dfs.items():
        plots.append(plot_metric_densities(exp_dfs, metrics, True, exp_name))
    plot = alt.vconcat(*plots).resolve_scale(y="shared", x="shared").properties(
        title='Metric Distributions Per Feature Description Method'
    ).configure_title(
        anchor='middle',
        fontSize=16,
        dy=-5,
    ).configure_axis(
        titleFontSize=13,
        titleColor="grey",
    ).configure_legend(
        orient='none',
        legendX=265,
        legendY=-50,
        direction='horizontal',
        titleBaseline='middle',
        titleOrient='left',
        labelFontSize=12,
        titleFontSize=12
    )
    return plot

In [27]:
distribution_plot = plot_metric_densities_multi(all_exp_dfs, metric_map.values())
save_chart(distribution_plot, 'metric_distributions.png');
# distribution_plot

## Compute Statistics on the Results

In [28]:
def paired_t_test(semantic_regex, baseline, delta, alpha):
    semantic_regex = np.asarray(semantic_regex, dtype=float)
    baseline = np.asarray(baseline, dtype=float)
    assert semantic_regex.shape == baseline.shape, "Input arrays must have the same shape"

    if delta > 0: # Non-inferiority test
        result = stats.ttest_rel(semantic_regex + delta, baseline, alternative="greater")
        test_type = "non-inferior"
    else: # Superiority test
        result = stats.ttest_rel(semantic_regex, baseline - delta, alternative="greater")
        test_type = "superior"

    passes = result.pvalue < alpha

    print(f"Paired t-test ({test_type}): t-statistic = {result.statistic:.4f}, p-value (one-sided) = {result.pvalue:.4g}")
    print(f"Conclusion: Semantic Regex is {test_type.upper() if passes else 'no claim'} to baseline by at least {delta}.")

    return result

### Non-Inferiority Test
Computes a one-sided paired t-test.
- *Question*: Are semantic regexes as good as natural language descriptions
- *Dependent variable*: Metric scores (continuous)
- *Independent variable*: Method (categorical)
- *Data*: Paired dataset samples (~1000). Given the number of samples, we expect the distributional differences to be normal.

In [29]:
def test_non_inferiority(exp_dfs, metrics, delta, alpha):
    for metric in metrics:
        print(f"\nNon-Inferiority Tests for Metric: {metric}".upper())
        semantic_regex_experiment_name = [name for name in exp_dfs.keys() if 'semantic' in name][0]
        semantic_regex_scores = exp_dfs[semantic_regex_experiment_name][metric]
        for experiment_name, experiment_df in exp_dfs.items():
            if experiment_name == semantic_regex_experiment_name:
                continue
            baseline_scores = experiment_df[metric]
            print(f"\nComparing {semantic_regex_experiment_name} to {experiment_name}")
            paired_t_test(semantic_regex_scores, baseline_scores, delta=delta, alpha=alpha)

In [30]:
delta = 0.05 # Non-inferiority margin
alpha = 0.05 # Significance level

for name, df in all_exp_dfs.items():
    print('\n' + '='*80 + '\n')
    print(name.upper())
    test_non_inferiority(df, metric_map.values(), delta, alpha)



GPT2-RES-25K

NON-INFERIORITY TESTS FOR METRIC: CLARITY

Comparing semantic-regex to token-act-pair
Paired t-test (non-inferior): t-statistic = 26.0228, p-value (one-sided) = 8.405e-121
Conclusion: Semantic Regex is NON-INFERIOR to baseline by at least 0.05.

Comparing semantic-regex to max-acts
Paired t-test (non-inferior): t-statistic = 4.2349, p-value (one-sided) = 1.224e-05
Conclusion: Semantic Regex is NON-INFERIOR to baseline by at least 0.05.

NON-INFERIORITY TESTS FOR METRIC: DETECT

Comparing semantic-regex to token-act-pair
Paired t-test (non-inferior): t-statistic = 21.7633, p-value (one-sided) = 4.361e-90
Conclusion: Semantic Regex is NON-INFERIOR to baseline by at least 0.05.

Comparing semantic-regex to max-acts
Paired t-test (non-inferior): t-statistic = 1.5421, p-value (one-sided) = 0.06165
Conclusion: Semantic Regex is no claim to baseline by at least 0.05.

NON-INFERIORITY TESTS FOR METRIC: FUZZING

Comparing semantic-regex to token-act-pair
Paired t-test (non-infer

### Superiority Tests
- *Question*: Are semantic regexes better than the natural language descriptions
- *Dependent variable*: Metric scores (continuous)
- *Independent variable*: Method (categorical)
- *Data*: Paired dataset samples (~1000). Given the number of samples, we expect the distributional differences to be normal.

In [31]:
def test_superiority(exp_dfs, metrics, delta, alpha):
    for metric in metrics:
        print(f"\nSuperiority Tests for Metric: {metric}".upper())
        semantic_regex_experiment_name = [name for name in exp_dfs.keys() if 'semantic' in name][0]
        semantic_regex_scores = exp_dfs[semantic_regex_experiment_name][metric]
        for experiment_name, experiment_df in exp_dfs.items():
            if experiment_name == semantic_regex_experiment_name:
                continue
            baseline_scores = experiment_df[metric]
            print(f"\nComparing {semantic_regex_experiment_name} to {experiment_name}")
            paired_t_test(semantic_regex_scores, baseline_scores, delta=delta, alpha=alpha)

In [32]:
delta = 0 # No margin in superiority test
alpha = 0.05 # Significance level

for name, df in all_exp_dfs.items():
    print('\n' + '='*80 + '\n')
    print(name.upper())
    num_baselines = len(df) - 1
    test_superiority(df, metric_map.values(), delta, alpha / num_baselines) # Significance level bonferroni correction for number



GPT2-RES-25K

SUPERIORITY TESTS FOR METRIC: CLARITY

Comparing semantic-regex to token-act-pair
Paired t-test (superior): t-statistic = 21.5320, p-value (one-sided) = 1.741e-88
Conclusion: Semantic Regex is SUPERIOR to baseline by at least 0.

Comparing semantic-regex to max-acts
Paired t-test (superior): t-statistic = -1.7757, p-value (one-sided) = 0.962
Conclusion: Semantic Regex is no claim to baseline by at least 0.

SUPERIORITY TESTS FOR METRIC: DETECT

Comparing semantic-regex to token-act-pair
Paired t-test (superior): t-statistic = 6.7694, p-value (one-sided) = 9.764e-12
Conclusion: Semantic Regex is SUPERIOR to baseline by at least 0.

Comparing semantic-regex to max-acts
Paired t-test (superior): t-statistic = -18.9070, p-value (one-sided) = 1
Conclusion: Semantic Regex is no claim to baseline by at least 0.

SUPERIORITY TESTS FOR METRIC: FUZZING

Comparing semantic-regex to token-act-pair
Paired t-test (superior): t-statistic = 5.7836, p-value (one-sided) = 4.574e-09
Concl

## Conciseness 

In [33]:
num_char_per_description = {}
for model, exp_dfs in all_exp_dfs.items():
    num_chars = {}
    for method, df in exp_dfs.items():
        desc_lengths = df['description'].apply(lambda x: len(x) if isinstance(x, str) else 0)
        avg_length = desc_lengths.mean()
        std_length = desc_lengths.std()
        num_chars[method] = (avg_length, std_length)
    num_char_per_description[model] = num_chars

for model, method_lengths in num_char_per_description.items():
    print(f"\nModel: {model.upper()}")
    for method, (avg_length, std_length) in method_lengths.items():
        print(f"Method: {method}, Average Length: {avg_length:.2f}, StdDev: {std_length:.2f}")


Model: GPT2-RES-25K
Method: token-act-pair, Average Length: 52.11, StdDev: 14.89
Method: max-acts, Average Length: 141.36, StdDev: 36.24
Method: semantic-regex, Average Length: 36.95, StdDev: 31.02

Model: GEMMA-2-2B-RES-16K
Method: token-act-pair, Average Length: 57.87, StdDev: 15.69
Method: max-acts, Average Length: 146.49, StdDev: 40.16
Method: semantic-regex, Average Length: 48.33, StdDev: 33.22

Model: GEMMA-2-2B-RES-65K
Method: token-act-pair, Average Length: 57.63, StdDev: 15.79
Method: max-acts, Average Length: 147.54, StdDev: 40.96
Method: semantic-regex, Average Length: 49.38, StdDev: 36.05


In [34]:
counts_per_method = {}
for model, exp_dfs in all_exp_dfs.items():
    num_chars = []
    for method, df in exp_dfs.items():
        desc_lengths = df['description'].apply(lambda x: len(x) if isinstance(x, str) else 0)
        if method not in counts_per_method:
            counts_per_method[method] = []
        counts_per_method[method].extend(desc_lengths.values)

for method, counts in counts_per_method.items():
    print(len(counts))
    q1, q3 = np.percentile(counts, [25, 75])
    print(f"Method: {method}, Average Length: {np.median(counts):.0f}, IDR {q1:.2f}-{q3:.2f}")

6500
Method: token-act-pair, Average Length: 55, IDR 46.00-66.00
6500
Method: max-acts, Average Length: 139, IDR 119.00-165.00
6500
Method: semantic-regex, Average Length: 40, IDR 19.00-58.00
